# **Redimension dNBR 2018-2021 10m à 1km**

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=================================================================================
SCRIPT ACADÉMIQUE : AGRÉGATION MULTI-BANDE ET ANALYSE DE SPILLOVER ÉCOLOGIQUE
Zone d'étude : Province du Sankuru, République Démocratique du Congo
Approche : Traitement par blocs fenêtrés avec Crop global de sécurité initial
Auteur : Dr. Daniel M.Y. Degina
=================================================================================
"""

import os
import sys
import subprocess
from pathlib import Path

# =================================================================================
# GESTION ET VÉRIFICATION AUTOMATIQUE DES DÉPENDANCES MANQUANTES
# =================================================================================
def _assurer_packages_installes():
    """
    Vérifie la présence des bibliothèques requises et procède à leur installation 
    via le gestionnaire pip en cas d'absence au sein de l'environnement d'exécution.
    """
    packages_requis = {
        "numpy": "numpy",
        "rasterio": "rasterio",
        "scipy": "scipy"
    }
    
    for module_name, package_name in packages_requis.items():
        try:
            __import__(module_name)
        except ImportError:
            print(f"[Gestion Dépendances] Le module '{module_name}' est absent. Installation en cours de '{package_name}'...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
                print(f"[Gestion Dépendances] '{package_name}' a été installé avec succès.")
            except Exception as e:
                print(f"[ERREUR CRITIQUE] Impossible d'installer le package requis '{package_name}' : {e}")
                sys.exit(1)

# Exécution de la vérification des packages en amont des imports principaux
_assurer_packages_installes()

# Importation sécurisée des bibliothèques scientifiques après validation de l'environnement
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy import stats

# =================================================================================
# FONCTION PRINCIPALE D'AGRÉGATION BLOC PAR BLOC (WINDOWED PROCESSING)
# =================================================================================
def aggregate_raster_by_quintiles(input_raster, output_raster, res_initiale, res_finale):
    """
    Agrège un raster continu (ex: dNBR / Déforestation) selon ses quintiles de la res_initiale à la res_finale.
    Calcule 3 bandes indépendantes par traitement fenêtré :
    Bande 1 : Classe dominante des quintiles (1 à 5)
    Bande 2 : Proportion de pixels appartenant au quintile 4 (décimal, ex: 0.234)
    Bande 3 : Proportion de pixels appartenant au quintile 5 (décimal, ex: 0.234)
    """
    print(f"\n[Début] Traitement optimisé du fichier : {input_raster.name}")
    
    # Validation du facteur d'échelle
    if res_finale % res_initiale != 0:
        raise ValueError(f"[ERREUR] La résolution finale ({res_finale}m) doit être un multiple strict de la résolution initiale ({res_initiale}m).")
    
    facteur = res_finale // res_initiale
    print(f"  Facteur d'agrégation calculé : {facteur} (Matrice locale de {facteur}x{facteur} pixels)")

    # Initialisation du générateur pseudo-aléatoire pour éliminer le biais spatial
    rng = np.random.default_rng(123)
    
    # -----------------------------------------------------------------------------
    # PASSE 1 : CALCUL DES LIMITES UTILES ET QUINTILES GLOBAUX
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 1 : COLLECTE DES STATISTIQUES GLOBALES (PASSE 1/2) ---")
    
    valeurs_pour_percentiles = []
    
    with rasterio.open(input_raster) as src:
        h_origine, l_origine = src.height, src.width
        meta_src = src.profile.copy()
        transform_src = src.transform
        crs_src = src.crs
        
        # Crop global unique : définition des dimensions utiles strictement multiples du facteur
        h_utile = (h_origine // facteur) * facteur
        l_utile = (l_origine // facteur) * facteur
        
        print(f"  Dimensions d'origine : {l_origine}x{h_origine}")
        print(f"  Dimensions utiles retenues (multiples de {facteur}) : {l_utile}x{h_utile}")
        print(f"  Bordure éliminée au départ (sans perte de blocs internes) : {l_origine - l_utile}px en X, {h_origine - h_utile}px en Y")
        
        # Définition d'une taille de bloc de lecture robuste et divisible par le facteur
        taille_bloc_y = 1000 if 1000 % facteur == 0 else facteur * 10
        taille_bloc_x = 1000 if 1000 % facteur == 0 else facteur * 10
        
        # Itération sur la zone utile uniquement
        for y in range(0, h_utile, taille_bloc_y):
            for x in range(0, l_utile, taille_bloc_x):
                largeur_w = min(taille_bloc_x, l_utile - x)
                hauteur_w = min(taille_bloc_y, h_utile - y)
                
                fenetre = Window(x, y, largeur_w, hauteur_w)
                donnees_bloc = src.read(1, window=fenetre, masked=True)
                
                valeurs_bloc = donnees_bloc.compressed()
                valeurs_bloc_finies = valeurs_bloc[np.isfinite(valeurs_bloc)]
                
                if valeurs_bloc_finies.size > 0:
                    # Échantillonnage aléatoire pour éviter le biais spatial
                    if valeurs_bloc_finies.size > 100000:
                        valeurs_bloc_finies = rng.choice(
                            valeurs_bloc_finies,
                            100000,
                            replace=False
                        )
                    valeurs_pour_percentiles.append(valeurs_bloc_finies)

    if len(valeurs_pour_percentiles) == 0:
        print(f"[Annulation] Aucun pixel valide fini trouvé dans la zone utile.")
        return
        
    valeurs_globales = np.concatenate(valeurs_pour_percentiles)
    del valeurs_pour_percentiles
    
    src_min = np.min(valeurs_globales)
    src_max = np.max(valeurs_globales)
    src_moy = np.mean(valeurs_globales)
    
    print(f"  Statistiques globales estimées sur la zone utile :")
    print(f"    - Minimum : {src_min:.4f}")
    print(f"    - Maximum : {src_max:.4f}")
    print(f"    - Moyenne : {src_moy:.4f}")

    # -----------------------------------------------------------------------------
    # ÉTAPE 2 : CALCUL DES SEUILS DE QUINTILES GLOBAUX
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 2 : SEUILS DES QUINTILES GLOBAUX ---")
    seuils = np.percentile(valeurs_globales, [20, 40, 60, 80])
    del valeurs_globales
    
    if np.any(np.isnan(seuils)):
        raise ValueError("[ERREUR CRITIQUE] Les seuils calculés contiennent des NaN.")
        
    print(f"  [OK] Confirmation : Aucun NaN détecté dans les seuils.")
    print(f"  Quintile 1 (0-20%)  : <= {seuils[0]:.4f}")
    print(f"  Quintile 2 (20-40%) : > {seuils[0]:.4f} et <= {seuils[1]:.4f}")
    print(f"  Quintile 3 (40-60%) : > {seuils[1]:.4f} et <= {seuils[2]:.4f}")
    print(f"  Quintile 4 (60-80%) : > {seuils[2]:.4f} et <= {seuils[3]:.4f}")
    print(f"  Quintile 5 (80-100%): > {seuils[3]:.4f}")

    # -----------------------------------------------------------------------------
    # ÉTAPE 3 : CONFIGURATION GÉOSPATIALE DU RASTER DE SORTIE
    # -----------------------------------------------------------------------------
    h_fin = h_utile // facteur
    l_fin = l_utile // facteur
    
    transform_fin = rasterio.Affine(
        transform_src.a * facteur, transform_src.b, transform_src.c,
        transform_src.d, transform_src.e * facteur, transform_src.f
    )
    
    meta_src.update({
        'driver': 'GTiff',
        'dtype': 'float32',  
        'nodata': -9999.0,
        'width': l_fin,
        'height': h_fin,
        'count': 3,
        'transform': transform_fin,
        'crs': crs_src,
        'compression': 'lzw'
    })

    # -----------------------------------------------------------------------------
    # ÉTAPE 4 : TRAITEMENT ET ÉCRITURE SANS PERTE PAR FENÊTRES (PASSE 2/2)
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 3 : AGRÉGATION MULTI-RÉSOLUTION BLOC PAR BLOC ---")
    
    dict_repartition_classes = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    somme_c2, nb_valides_c2 = 0.0, 0
    somme_c3, nb_valides_c3 = 0.0, 0
    
    # Taille fixe pour l'analyse par blocs (ex: 1000x1000 pixels d'origine)
    taille_fenetre_y = 1000 if 1000 % facteur == 0 else facteur * 10
    taille_fenetre_x = 1000 if 1000 % facteur == 0 else facteur * 10
    
    with rasterio.open(input_raster) as src, rasterio.open(output_raster, 'w', **meta_src) as dst:
        
        dst.set_band_description(1, "Classe dominante du quintile (1-5)")
        dst.set_band_description(2, "Proportion du quintile Q4 (0-1)")
        dst.set_band_description(3, "Proportion du quintile Q5 (0-1)")
        
        for y in range(0, h_utile, taille_fenetre_y):
            for x in range(0, l_utile, taille_fenetre_x):
                
                largeur_src = min(taille_fenetre_x, l_utile - x)
                hauteur_src = min(taille_fenetre_y, h_utile - y)
                
                # Vérifications structurelles impératives avant instanciation et reshape
                assert hauteur_src % facteur == 0, f"[Assertion Échouée] La hauteur du bloc ({hauteur_src}) n'est pas un multiple de {facteur}."
                assert largeur_src % facteur == 0, f"[Assertion Échouée] La largeur du bloc ({largeur_src}) n'est pas un multiple de {facteur}."
                
                fenetre_src = Window(x, y, largeur_src, hauteur_src)
                pop_10m_bloc = src.read(1, window=fenetre_src, masked=True)
                
                # Classification catégorielle locale
                classes_bloc = np.zeros(pop_10m_bloc.shape, dtype=np.uint8)
                mask_pixels_calculables = (~pop_10m_bloc.mask) & np.isfinite(pop_10m_bloc.data)
                
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data <= seuils[0])] = 1
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[0]) & (pop_10m_bloc.data <= seuils[1])] = 2
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[1]) & (pop_10m_bloc.data <= seuils[2])] = 3
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[2]) & (pop_10m_bloc.data <= seuils[3])] = 4
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[3])] = 5
                
                classes_bloc_mask = (~mask_pixels_calculables) | (classes_bloc == 0)
                classes_bloc_ma = np.ma.array(classes_bloc, mask=classes_bloc_mask)
                
                h_fin_bloc = hauteur_src // facteur
                l_fin_bloc = largeur_src // facteur
                
                out_classe_dominante = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                out_prop_Q4 = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                out_prop_Q5 = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                
                blocs_classes = classes_bloc_ma.reshape(h_fin_bloc, facteur, l_fin_bloc, facteur)
                
                for i in range(h_fin_bloc):
                    for j in range(l_fin_bloc):
                        p_class = blocs_classes[i, :, j, :].compressed()
                        p_class = p_class[p_class > 0]
                        
                        nb_valides = len(p_class)
                        
                        if nb_valides > 0:
                            mode_res = stats.mode(p_class, keepdims=True)
                            val_classe_dom = mode_res.mode[0]
                            out_classe_dominante[i, j] = val_classe_dom
                            
                            # MODIFICATION : Calcul en proportion décimale (0 à 1) au lieu de pourcentage
                            val_prop_Q4 = np.sum(p_class == 4) / nb_valides
                            out_prop_Q4[i, j] = val_prop_Q4
                            
                            val_prop_Q5 = np.sum(p_class == 5) / nb_valides
                            out_prop_Q5[i, j] = val_prop_Q5
                            
                            if val_classe_dom in dict_repartition_classes:
                                dict_repartition_classes[val_classe_dom] += 1
                            
                            somme_c2 += val_prop_Q4
                            nb_valides_c2 += 1
                            
                            somme_c3 += val_prop_Q5
                            nb_valides_c3 += 1
                
                fenetre_dst = Window(x // facteur, y // facteur, l_fin_bloc, h_fin_bloc)
                
                dst.write(out_classe_dominante, 1, window=fenetre_dst)
                dst.write(out_prop_Q4, 2, window=fenetre_dst)
                dst.write(out_prop_Q5, 3, window=fenetre_dst)

    # -----------------------------------------------------------------------------
    # ÉTAPE 5 : CONTRÔLES ET VÉRIFICATIONS STRUCTURELLES FINALES
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 4 : CONTRÔLES ET SÉCURITÉ GÉOSPATIALE ---")
    with rasterio.open(output_raster) as img_verif:
        print(f"  [VÉRIFICATION] Nombre de bandes générées : {img_verif.count}")
        
        print("\n--- ÉTAPE 5 : STATISTIQUES DESCRIPTIVES DU RASTER AGRÉGÉ ---")
        print("  Répartition de la variable 'classe_dominante' (Bande 1) :")
        for classe_id, total_pixels in dict_repartition_classes.items():
            if total_pixels > 0:
                print(f"    - Classe {classe_id} : {total_pixels} pixels")
            
        moyenne_spatiale_c2 = (somme_c2 / nb_valides_c2) if nb_valides_c2 > 0 else 0.0
        moyenne_spatiale_c3 = (somme_c3 / nb_valides_c3) if nb_valides_c3 > 0 else 0.0
        
        print(f"  Moyenne spatiale de 'prop_Q4' (Bande 2) : {moyenne_spatiale_c2:.4f}")
        print(f"  Moyenne spatiale de 'prop_Q5' (Bande 3) : {moyenne_spatiale_c3:.4f}")
        
    print(f"[Succès] Fichier sauvegardé sans aucune perte de bloc : {output_raster.resolve()}\n")

# ==========================================
# EXÉCUTION DYNAMIQUE
# ==========================================
if __name__ == "__main__":
    # Paramétrage des résolutions initiale et finale (ex: 10m à 1000m / 1km)
    resinit = 10
    resfin = 1000
    
    dir_input = Path("../geodata_outputs")
    dir_output = Path("../geodata_outputs")
    
    dir_output.mkdir(parents=True, exist_ok=True)
    
    # Configuration des répertoires pour le fichier de déforestation
    input_file = dir_input / "deforest_2018_2021.tif"
    output_file = dir_output / "deforest_2018_2021_redim_1km_class_propDefor.tif"
    
    if input_file.exists():
        aggregate_raster_by_quintiles(input_file, output_file, resinit, resfin)
    else:
        print(f"[Erreur] Le fichier d'entrée spécifié est introuvable : {input_file.resolve()}")


[Début] Traitement optimisé du fichier : deforest_2018_2021.tif
  Facteur d'agrégation calculé : 100 (Matrice locale de 100x100 pixels)
--- ÉTAPE 1 : COLLECTE DES STATISTIQUES GLOBALES (PASSE 1/2) ---
  Dimensions d'origine : 7928x8393
  Dimensions utiles retenues (multiples de 100) : 7900x8300
  Bordure éliminée au départ (sans perte de blocs internes) : 28px en X, 93px en Y
  Statistiques globales estimées sur la zone utile :
    - Minimum : -1.2070
    - Maximum : 1.2954
    - Moyenne : 0.0301
--- ÉTAPE 2 : SEUILS DES QUINTILES GLOBAUX ---
  [OK] Confirmation : Aucun NaN détecté dans les seuils.
  Quintile 1 (0-20%)  : <= 0.0016
  Quintile 2 (20-40%) : > 0.0016 et <= 0.0239
  Quintile 3 (40-60%) : > 0.0239 et <= 0.0408
  Quintile 4 (60-80%) : > 0.0408 et <= 0.0609
  Quintile 5 (80-100%): > 0.0609
--- ÉTAPE 3 : AGRÉGATION MULTI-RÉSOLUTION BLOC PAR BLOC ---
--- ÉTAPE 4 : CONTRÔLES ET SÉCURITÉ GÉOSPATIALE ---
  [VÉRIFICATION] Nombre de bandes générées : 3

--- ÉTAPE 5 : STATISTIQUES D

# **Redimension dNBR 2019-2021 10m à 1km**

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=================================================================================
SCRIPT ACADÉMIQUE : AGRÉGATION MULTI-BANDE ET ANALYSE DE SPILLOVER ÉCOLOGIQUE
Zone d'étude : Province du Sankuru, République Démocratique du Congo
Approche : Traitement par blocs fenêtrés avec Crop global de sécurité initial
Auteur : Dr. Daniel M.Y. Degina
=================================================================================
"""

import os
import sys
import subprocess
from pathlib import Path

# =================================================================================
# GESTION ET VÉRIFICATION AUTOMATIQUE DES DÉPENDANCES MANQUANTES
# =================================================================================
def _assurer_packages_installes():
    """
    Vérifie la présence des bibliothèques requises et procède à leur installation 
    via le gestionnaire pip en cas d'absence au sein de l'environnement d'exécution.
    """
    packages_requis = {
        "numpy": "numpy",
        "rasterio": "rasterio",
        "scipy": "scipy"
    }
    
    for module_name, package_name in packages_requis.items():
        try:
            __import__(module_name)
        except ImportError:
            print(f"[Gestion Dépendances] Le module '{module_name}' est absent. Installation en cours de '{package_name}'...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
                print(f"[Gestion Dépendances] '{package_name}' a été installé avec succès.")
            except Exception as e:
                print(f"[ERREUR CRITIQUE] Impossible d'installer le package requis '{package_name}' : {e}")
                sys.exit(1)

# Exécution de la vérification des packages en amont des imports principaux
_assurer_packages_installes()

# Importation sécurisée des bibliothèques scientifiques après validation de l'environnement
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy import stats

# =================================================================================
# FONCTION PRINCIPALE D'AGRÉGATION BLOC PAR BLOC (WINDOWED PROCESSING)
# =================================================================================
def aggregate_raster_by_quintiles(input_raster, output_raster, res_initiale, res_finale):
    """
    Agrège un raster continu (ex: dNBR / Déforestation) selon ses quintiles de la res_initiale à la res_finale.
    Calcule 3 bandes indépendantes par traitement fenêtré :
    Bande 1 : Classe dominante des quintiles (1 à 5)
    Bande 2 : Proportion de pixels appartenant au quintile 4 (décimal, ex: 0.234)
    Bande 3 : Proportion de pixels appartenant au quintile 5 (décimal, ex: 0.234)
    """
    print(f"\n[Début] Traitement optimisé du fichier : {input_raster.name}")
    
    # Validation du facteur d'échelle
    if res_finale % res_initiale != 0:
        raise ValueError(f"[ERREUR] La résolution finale ({res_finale}m) doit être un multiple strict de la résolution initiale ({res_initiale}m).")
    
    facteur = res_finale // res_initiale
    print(f"  Facteur d'agrégation calculé : {facteur} (Matrice locale de {facteur}x{facteur} pixels)")

    # Initialisation du générateur pseudo-aléatoire pour éliminer le biais spatial
    rng = np.random.default_rng(123)
    
    # -----------------------------------------------------------------------------
    # PASSE 1 : CALCUL DES LIMITES UTILES ET QUINTILES GLOBAUX
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 1 : COLLECTE DES STATISTIQUES GLOBALES (PASSE 1/2) ---")
    
    valeurs_pour_percentiles = []
    
    with rasterio.open(input_raster) as src:
        h_origine, l_origine = src.height, src.width
        meta_src = src.profile.copy()
        transform_src = src.transform
        crs_src = src.crs
        
        # Crop global unique : définition des dimensions utiles strictement multiples du facteur
        h_utile = (h_origine // facteur) * facteur
        l_utile = (l_origine // facteur) * facteur
        
        print(f"  Dimensions d'origine : {l_origine}x{h_origine}")
        print(f"  Dimensions utiles retenues (multiples de {facteur}) : {l_utile}x{h_utile}")
        print(f"  Bordure éliminée au départ (sans perte de blocs internes) : {l_origine - l_utile}px en X, {h_origine - h_utile}px en Y")
        
        # Définition d'une taille de bloc de lecture robuste et divisible par le facteur
        taille_bloc_y = 1000 if 1000 % facteur == 0 else facteur * 10
        taille_bloc_x = 1000 if 1000 % facteur == 0 else facteur * 10
        
        # Itération sur la zone utile uniquement
        for y in range(0, h_utile, taille_bloc_y):
            for x in range(0, l_utile, taille_bloc_x):
                largeur_w = min(taille_bloc_x, l_utile - x)
                hauteur_w = min(taille_bloc_y, h_utile - y)
                
                fenetre = Window(x, y, largeur_w, hauteur_w)
                donnees_bloc = src.read(1, window=fenetre, masked=True)
                
                valeurs_bloc = donnees_bloc.compressed()
                valeurs_bloc_finies = valeurs_bloc[np.isfinite(valeurs_bloc)]
                
                if valeurs_bloc_finies.size > 0:
                    # Échantillonnage aléatoire pour éviter le biais spatial
                    if valeurs_bloc_finies.size > 100000:
                        valeurs_bloc_finies = rng.choice(
                            valeurs_bloc_finies,
                            100000,
                            replace=False
                        )
                    valeurs_pour_percentiles.append(valeurs_bloc_finies)

    if len(valeurs_pour_percentiles) == 0:
        print(f"[Annulation] Aucun pixel valide fini trouvé dans la zone utile.")
        return
        
    valeurs_globales = np.concatenate(valeurs_pour_percentiles)
    del valeurs_pour_percentiles
    
    src_min = np.min(valeurs_globales)
    src_max = np.max(valeurs_globales)
    src_moy = np.mean(valeurs_globales)
    
    print(f"  Statistiques globales estimées sur la zone utile :")
    print(f"    - Minimum : {src_min:.4f}")
    print(f"    - Maximum : {src_max:.4f}")
    print(f"    - Moyenne : {src_moy:.4f}")

    # -----------------------------------------------------------------------------
    # ÉTAPE 2 : CALCUL DES SEUILS DE QUINTILES GLOBAUX
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 2 : SEUILS DES QUINTILES GLOBAUX ---")
    seuils = np.percentile(valeurs_globales, [20, 40, 60, 80])
    del valeurs_globales
    
    if np.any(np.isnan(seuils)):
        raise ValueError("[ERREUR CRITIQUE] Les seuils calculés contiennent des NaN.")
        
    print(f"  [OK] Confirmation : Aucun NaN détecté dans les seuils.")
    print(f"  Quintile 1 (0-20%)  : <= {seuils[0]:.4f}")
    print(f"  Quintile 2 (20-40%) : > {seuils[0]:.4f} et <= {seuils[1]:.4f}")
    print(f"  Quintile 3 (40-60%) : > {seuils[1]:.4f} et <= {seuils[2]:.4f}")
    print(f"  Quintile 4 (60-80%) : > {seuils[2]:.4f} et <= {seuils[3]:.4f}")
    print(f"  Quintile 5 (80-100%): > {seuils[3]:.4f}")

    # -----------------------------------------------------------------------------
    # ÉTAPE 3 : CONFIGURATION GÉOSPATIALE DU RASTER DE SORTIE
    # -----------------------------------------------------------------------------
    h_fin = h_utile // facteur
    l_fin = l_utile // facteur
    
    transform_fin = rasterio.Affine(
        transform_src.a * facteur, transform_src.b, transform_src.c,
        transform_src.d, transform_src.e * facteur, transform_src.f
    )
    
    meta_src.update({
        'driver': 'GTiff',
        'dtype': 'float32',  
        'nodata': -9999.0,
        'width': l_fin,
        'height': h_fin,
        'count': 3,
        'transform': transform_fin,
        'crs': crs_src,
        'compression': 'lzw'
    })

    # -----------------------------------------------------------------------------
    # ÉTAPE 4 : TRAITEMENT ET ÉCRITURE SANS PERTE PAR FENÊTRES (PASSE 2/2)
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 3 : AGRÉGATION MULTI-RÉSOLUTION BLOC PAR BLOC ---")
    
    dict_repartition_classes = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    somme_c2, nb_valides_c2 = 0.0, 0
    somme_c3, nb_valides_c3 = 0.0, 0
    
    # Taille fixe pour l'analyse par blocs (ex: 1000x1000 pixels d'origine)
    taille_fenetre_y = 1000 if 1000 % facteur == 0 else facteur * 10
    taille_fenetre_x = 1000 if 1000 % facteur == 0 else facteur * 10
    
    with rasterio.open(input_raster) as src, rasterio.open(output_raster, 'w', **meta_src) as dst:
        
        dst.set_band_description(1, "Classe dominante du quintile (1-5)")
        dst.set_band_description(2, "Proportion du quintile Q4 (0-1)")
        dst.set_band_description(3, "Proportion du quintile Q5 (0-1)")
        
        for y in range(0, h_utile, taille_fenetre_y):
            for x in range(0, l_utile, taille_fenetre_x):
                
                largeur_src = min(taille_fenetre_x, l_utile - x)
                hauteur_src = min(taille_fenetre_y, h_utile - y)
                
                # Vérifications structurelles impératives avant instanciation et reshape
                assert hauteur_src % facteur == 0, f"[Assertion Échouée] La hauteur du bloc ({hauteur_src}) n'est pas un multiple de {facteur}."
                assert largeur_src % facteur == 0, f"[Assertion Échouée] La largeur du bloc ({largeur_src}) n'est pas un multiple de {facteur}."
                
                fenetre_src = Window(x, y, largeur_src, hauteur_src)
                pop_10m_bloc = src.read(1, window=fenetre_src, masked=True)
                
                # Classification catégorielle locale
                classes_bloc = np.zeros(pop_10m_bloc.shape, dtype=np.uint8)
                mask_pixels_calculables = (~pop_10m_bloc.mask) & np.isfinite(pop_10m_bloc.data)
                
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data <= seuils[0])] = 1
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[0]) & (pop_10m_bloc.data <= seuils[1])] = 2
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[1]) & (pop_10m_bloc.data <= seuils[2])] = 3
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[2]) & (pop_10m_bloc.data <= seuils[3])] = 4
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[3])] = 5
                
                classes_bloc_mask = (~mask_pixels_calculables) | (classes_bloc == 0)
                classes_bloc_ma = np.ma.array(classes_bloc, mask=classes_bloc_mask)
                
                h_fin_bloc = hauteur_src // facteur
                l_fin_bloc = largeur_src // facteur
                
                out_classe_dominante = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                out_prop_Q4 = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                out_prop_Q5 = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                
                blocs_classes = classes_bloc_ma.reshape(h_fin_bloc, facteur, l_fin_bloc, facteur)
                
                for i in range(h_fin_bloc):
                    for j in range(l_fin_bloc):
                        p_class = blocs_classes[i, :, j, :].compressed()
                        p_class = p_class[p_class > 0]
                        
                        nb_valides = len(p_class)
                        
                        if nb_valides > 0:
                            mode_res = stats.mode(p_class, keepdims=True)
                            val_classe_dom = mode_res.mode[0]
                            out_classe_dominante[i, j] = val_classe_dom
                            
                            # MODIFICATION : Calcul en proportion décimale (0 à 1) au lieu de pourcentage
                            val_prop_Q4 = np.sum(p_class == 4) / nb_valides
                            out_prop_Q4[i, j] = val_prop_Q4
                            
                            val_prop_Q5 = np.sum(p_class == 5) / nb_valides
                            out_prop_Q5[i, j] = val_prop_Q5
                            
                            if val_classe_dom in dict_repartition_classes:
                                dict_repartition_classes[val_classe_dom] += 1
                            
                            somme_c2 += val_prop_Q4
                            nb_valides_c2 += 1
                            
                            somme_c3 += val_prop_Q5
                            nb_valides_c3 += 1
                
                fenetre_dst = Window(x // facteur, y // facteur, l_fin_bloc, h_fin_bloc)
                
                dst.write(out_classe_dominante, 1, window=fenetre_dst)
                dst.write(out_prop_Q4, 2, window=fenetre_dst)
                dst.write(out_prop_Q5, 3, window=fenetre_dst)

    # -----------------------------------------------------------------------------
    # ÉTAPE 5 : CONTRÔLES ET VÉRIFICATIONS STRUCTURELLES FINALES
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 4 : CONTRÔLES ET SÉCURITÉ GÉOSPATIALE ---")
    with rasterio.open(output_raster) as img_verif:
        print(f"  [VÉRIFICATION] Nombre de bandes générées : {img_verif.count}")
        
        print("\n--- ÉTAPE 5 : STATISTIQUES DESCRIPTIVES DU RASTER AGRÉGÉ ---")
        print("  Répartition de la variable 'classe_dominante' (Bande 1) :")
        for classe_id, total_pixels in dict_repartition_classes.items():
            if total_pixels > 0:
                print(f"    - Classe {classe_id} : {total_pixels} pixels")
            
        moyenne_spatiale_c2 = (somme_c2 / nb_valides_c2) if nb_valides_c2 > 0 else 0.0
        moyenne_spatiale_c3 = (somme_c3 / nb_valides_c3) if nb_valides_c3 > 0 else 0.0
        
        print(f"  Moyenne spatiale de 'prop_Q4' (Bande 2) : {moyenne_spatiale_c2:.4f}")
        print(f"  Moyenne spatiale de 'prop_Q5' (Bande 3) : {moyenne_spatiale_c3:.4f}")
        
    print(f"[Succès] Fichier sauvegardé sans aucune perte de bloc : {output_raster.resolve()}\n")

# ==========================================
# EXÉCUTION DYNAMIQUE
# ==========================================
if __name__ == "__main__":
    # Paramétrage des résolutions initiale et finale (ex: 10m à 1000m / 1km)
    resinit = 10
    resfin = 1000
    
    dir_input = Path("../geodata_outputs")
    dir_output = Path("../geodata_outputs")
    
    dir_output.mkdir(parents=True, exist_ok=True)
    
    # Configuration des répertoires pour le fichier de déforestation
    input_file = dir_input / "deforest_2019_2021.tif"
    output_file = dir_output / "deforest_2019_2021_redim_1km_class_propDefor.tif"
    
    if input_file.exists():
        aggregate_raster_by_quintiles(input_file, output_file, resinit, resfin)
    else:
        print(f"[Erreur] Le fichier d'entrée spécifié est introuvable : {input_file.resolve()}")


[Début] Traitement optimisé du fichier : deforest_2019_2021.tif
  Facteur d'agrégation calculé : 100 (Matrice locale de 100x100 pixels)
--- ÉTAPE 1 : COLLECTE DES STATISTIQUES GLOBALES (PASSE 1/2) ---
  Dimensions d'origine : 7928x8393
  Dimensions utiles retenues (multiples de 100) : 7900x8300
  Bordure éliminée au départ (sans perte de blocs internes) : 28px en X, 93px en Y
  Statistiques globales estimées sur la zone utile :
    - Minimum : -0.9317
    - Maximum : 1.0938
    - Moyenne : 0.0005
--- ÉTAPE 2 : SEUILS DES QUINTILES GLOBAUX ---
  [OK] Confirmation : Aucun NaN détecté dans les seuils.
  Quintile 1 (0-20%)  : <= -0.0252
  Quintile 2 (20-40%) : > -0.0252 et <= -0.0092
  Quintile 3 (40-60%) : > -0.0092 et <= 0.0042
  Quintile 4 (60-80%) : > 0.0042 et <= 0.0224
  Quintile 5 (80-100%): > 0.0224
--- ÉTAPE 3 : AGRÉGATION MULTI-RÉSOLUTION BLOC PAR BLOC ---
--- ÉTAPE 4 : CONTRÔLES ET SÉCURITÉ GÉOSPATIALE ---
  [VÉRIFICATION] Nombre de bandes générées : 3

--- ÉTAPE 5 : STATISTIQU

# **Redimension dNBR 2020-2021 10m à 1km**

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
=================================================================================
SCRIPT ACADÉMIQUE : AGRÉGATION MULTI-BANDE ET ANALYSE DE SPILLOVER ÉCOLOGIQUE
Zone d'étude : Province du Sankuru, République Démocratique du Congo
Approche : Traitement par blocs fenêtrés avec Crop global de sécurité initial
Auteur : Dr. Daniel M.Y. Degina
=================================================================================
"""

import os
import sys
import subprocess
from pathlib import Path

# =================================================================================
# GESTION ET VÉRIFICATION AUTOMATIQUE DES DÉPENDANCES MANQUANTES
# =================================================================================
def _assurer_packages_installes():
    """
    Vérifie la présence des bibliothèques requises et procède à leur installation 
    via le gestionnaire pip en cas d'absence au sein de l'environnement d'exécution.
    """
    packages_requis = {
        "numpy": "numpy",
        "rasterio": "rasterio",
        "scipy": "scipy"
    }
    
    for module_name, package_name in packages_requis.items():
        try:
            __import__(module_name)
        except ImportError:
            print(f"[Gestion Dépendances] Le module '{module_name}' est absent. Installation en cours de '{package_name}'...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
                print(f"[Gestion Dépendances] '{package_name}' a été installé avec succès.")
            except Exception as e:
                print(f"[ERREUR CRITIQUE] Impossible d'installer le package requis '{package_name}' : {e}")
                sys.exit(1)

# Exécution de la vérification des packages en amont des imports principaux
_assurer_packages_installes()

# Importation sécurisée des bibliothèques scientifiques après validation de l'environnement
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy import stats

# =================================================================================
# FONCTION PRINCIPALE D'AGRÉGATION BLOC PAR BLOC (WINDOWED PROCESSING)
# =================================================================================
def aggregate_raster_by_quintiles(input_raster, output_raster, res_initiale, res_finale):
    """
    Agrège un raster continu (ex: dNBR / Déforestation) selon ses quintiles de la res_initiale à la res_finale.
    Calcule 3 bandes indépendantes par traitement fenêtré :
    Bande 1 : Classe dominante des quintiles (1 à 5)
    Bande 2 : Proportion de pixels appartenant au quintile 4 (décimal, ex: 0.234)
    Bande 3 : Proportion de pixels appartenant au quintile 5 (décimal, ex: 0.234)
    """
    print(f"\n[Début] Traitement optimisé du fichier : {input_raster.name}")
    
    # Validation du facteur d'échelle
    if res_finale % res_initiale != 0:
        raise ValueError(f"[ERREUR] La résolution finale ({res_finale}m) doit être un multiple strict de la résolution initiale ({res_initiale}m).")
    
    facteur = res_finale // res_initiale
    print(f"  Facteur d'agrégation calculé : {facteur} (Matrice locale de {facteur}x{facteur} pixels)")

    # Initialisation du générateur pseudo-aléatoire pour éliminer le biais spatial
    rng = np.random.default_rng(123)
    
    # -----------------------------------------------------------------------------
    # PASSE 1 : CALCUL DES LIMITES UTILES ET QUINTILES GLOBAUX
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 1 : COLLECTE DES STATISTIQUES GLOBALES (PASSE 1/2) ---")
    
    valeurs_pour_percentiles = []
    
    with rasterio.open(input_raster) as src:
        h_origine, l_origine = src.height, src.width
        meta_src = src.profile.copy()
        transform_src = src.transform
        crs_src = src.crs
        
        # Crop global unique : définition des dimensions utiles strictement multiples du facteur
        h_utile = (h_origine // facteur) * facteur
        l_utile = (l_origine // facteur) * facteur
        
        print(f"  Dimensions d'origine : {l_origine}x{h_origine}")
        print(f"  Dimensions utiles retenues (multiples de {facteur}) : {l_utile}x{h_utile}")
        print(f"  Bordure éliminée au départ (sans perte de blocs internes) : {l_origine - l_utile}px en X, {h_origine - h_utile}px en Y")
        
        # Définition d'une taille de bloc de lecture robuste et divisible par le facteur
        taille_bloc_y = 1000 if 1000 % facteur == 0 else facteur * 10
        taille_bloc_x = 1000 if 1000 % facteur == 0 else facteur * 10
        
        # Itération sur la zone utile uniquement
        for y in range(0, h_utile, taille_bloc_y):
            for x in range(0, l_utile, taille_bloc_x):
                largeur_w = min(taille_bloc_x, l_utile - x)
                hauteur_w = min(taille_bloc_y, h_utile - y)
                
                fenetre = Window(x, y, largeur_w, hauteur_w)
                donnees_bloc = src.read(1, window=fenetre, masked=True)
                
                valeurs_bloc = donnees_bloc.compressed()
                valeurs_bloc_finies = valeurs_bloc[np.isfinite(valeurs_bloc)]
                
                if valeurs_bloc_finies.size > 0:
                    # Échantillonnage aléatoire pour éviter le biais spatial
                    if valeurs_bloc_finies.size > 100000:
                        valeurs_bloc_finies = rng.choice(
                            valeurs_bloc_finies,
                            100000,
                            replace=False
                        )
                    valeurs_pour_percentiles.append(valeurs_bloc_finies)

    if len(valeurs_pour_percentiles) == 0:
        print(f"[Annulation] Aucun pixel valide fini trouvé dans la zone utile.")
        return
        
    valeurs_globales = np.concatenate(valeurs_pour_percentiles)
    del valeurs_pour_percentiles
    
    src_min = np.min(valeurs_globales)
    src_max = np.max(valeurs_globales)
    src_moy = np.mean(valeurs_globales)
    
    print(f"  Statistiques globales estimées sur la zone utile :")
    print(f"    - Minimum : {src_min:.4f}")
    print(f"    - Maximum : {src_max:.4f}")
    print(f"    - Moyenne : {src_moy:.4f}")

    # -----------------------------------------------------------------------------
    # ÉTAPE 2 : CALCUL DES SEUILS DE QUINTILES GLOBAUX
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 2 : SEUILS DES QUINTILES GLOBAUX ---")
    seuils = np.percentile(valeurs_globales, [20, 40, 60, 80])
    del valeurs_globales
    
    if np.any(np.isnan(seuils)):
        raise ValueError("[ERREUR CRITIQUE] Les seuils calculés contiennent des NaN.")
        
    print(f"  [OK] Confirmation : Aucun NaN détecté dans les seuils.")
    print(f"  Quintile 1 (0-20%)  : <= {seuils[0]:.4f}")
    print(f"  Quintile 2 (20-40%) : > {seuils[0]:.4f} et <= {seuils[1]:.4f}")
    print(f"  Quintile 3 (40-60%) : > {seuils[1]:.4f} et <= {seuils[2]:.4f}")
    print(f"  Quintile 4 (60-80%) : > {seuils[2]:.4f} et <= {seuils[3]:.4f}")
    print(f"  Quintile 5 (80-100%): > {seuils[3]:.4f}")

    # -----------------------------------------------------------------------------
    # ÉTAPE 3 : CONFIGURATION GÉOSPATIALE DU RASTER DE SORTIE
    # -----------------------------------------------------------------------------
    h_fin = h_utile // facteur
    l_fin = l_utile // facteur
    
    transform_fin = rasterio.Affine(
        transform_src.a * facteur, transform_src.b, transform_src.c,
        transform_src.d, transform_src.e * facteur, transform_src.f
    )
    
    meta_src.update({
        'driver': 'GTiff',
        'dtype': 'float32',  
        'nodata': -9999.0,
        'width': l_fin,
        'height': h_fin,
        'count': 3,
        'transform': transform_fin,
        'crs': crs_src,
        'compression': 'lzw'
    })

    # -----------------------------------------------------------------------------
    # ÉTAPE 4 : TRAITEMENT ET ÉCRITURE SANS PERTE PAR FENÊTRES (PASSE 2/2)
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 3 : AGRÉGATION MULTI-RÉSOLUTION BLOC PAR BLOC ---")
    
    dict_repartition_classes = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    somme_c2, nb_valides_c2 = 0.0, 0
    somme_c3, nb_valides_c3 = 0.0, 0
    
    # Taille fixe pour l'analyse par blocs (ex: 1000x1000 pixels d'origine)
    taille_fenetre_y = 1000 if 1000 % facteur == 0 else facteur * 10
    taille_fenetre_x = 1000 if 1000 % facteur == 0 else facteur * 10
    
    with rasterio.open(input_raster) as src, rasterio.open(output_raster, 'w', **meta_src) as dst:
        
        dst.set_band_description(1, "Classe dominante du quintile (1-5)")
        dst.set_band_description(2, "Proportion du quintile Q4 (0-1)")
        dst.set_band_description(3, "Proportion du quintile Q5 (0-1)")
        
        for y in range(0, h_utile, taille_fenetre_y):
            for x in range(0, l_utile, taille_fenetre_x):
                
                largeur_src = min(taille_fenetre_x, l_utile - x)
                hauteur_src = min(taille_fenetre_y, h_utile - y)
                
                # Vérifications structurelles impératives avant instanciation et reshape
                assert hauteur_src % facteur == 0, f"[Assertion Échouée] La hauteur du bloc ({hauteur_src}) n'est pas un multiple de {facteur}."
                assert largeur_src % facteur == 0, f"[Assertion Échouée] La largeur du bloc ({largeur_src}) n'est pas un multiple de {facteur}."
                
                fenetre_src = Window(x, y, largeur_src, hauteur_src)
                pop_10m_bloc = src.read(1, window=fenetre_src, masked=True)
                
                # Classification catégorielle locale
                classes_bloc = np.zeros(pop_10m_bloc.shape, dtype=np.uint8)
                mask_pixels_calculables = (~pop_10m_bloc.mask) & np.isfinite(pop_10m_bloc.data)
                
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data <= seuils[0])] = 1
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[0]) & (pop_10m_bloc.data <= seuils[1])] = 2
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[1]) & (pop_10m_bloc.data <= seuils[2])] = 3
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[2]) & (pop_10m_bloc.data <= seuils[3])] = 4
                classes_bloc[mask_pixels_calculables & (pop_10m_bloc.data > seuils[3])] = 5
                
                classes_bloc_mask = (~mask_pixels_calculables) | (classes_bloc == 0)
                classes_bloc_ma = np.ma.array(classes_bloc, mask=classes_bloc_mask)
                
                h_fin_bloc = hauteur_src // facteur
                l_fin_bloc = largeur_src // facteur
                
                out_classe_dominante = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                out_prop_Q4 = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                out_prop_Q5 = np.full((h_fin_bloc, l_fin_bloc), -9999.0, dtype=np.float32)
                
                blocs_classes = classes_bloc_ma.reshape(h_fin_bloc, facteur, l_fin_bloc, facteur)
                
                for i in range(h_fin_bloc):
                    for j in range(l_fin_bloc):
                        p_class = blocs_classes[i, :, j, :].compressed()
                        p_class = p_class[p_class > 0]
                        
                        nb_valides = len(p_class)
                        
                        if nb_valides > 0:
                            mode_res = stats.mode(p_class, keepdims=True)
                            val_classe_dom = mode_res.mode[0]
                            out_classe_dominante[i, j] = val_classe_dom
                            
                            # MODIFICATION : Calcul en proportion décimale (0 à 1) au lieu de pourcentage
                            val_prop_Q4 = np.sum(p_class == 4) / nb_valides
                            out_prop_Q4[i, j] = val_prop_Q4
                            
                            val_prop_Q5 = np.sum(p_class == 5) / nb_valides
                            out_prop_Q5[i, j] = val_prop_Q5
                            
                            if val_classe_dom in dict_repartition_classes:
                                dict_repartition_classes[val_classe_dom] += 1
                            
                            somme_c2 += val_prop_Q4
                            nb_valides_c2 += 1
                            
                            somme_c3 += val_prop_Q5
                            nb_valides_c3 += 1
                
                fenetre_dst = Window(x // facteur, y // facteur, l_fin_bloc, h_fin_bloc)
                
                dst.write(out_classe_dominante, 1, window=fenetre_dst)
                dst.write(out_prop_Q4, 2, window=fenetre_dst)
                dst.write(out_prop_Q5, 3, window=fenetre_dst)

    # -----------------------------------------------------------------------------
    # ÉTAPE 5 : CONTRÔLES ET VÉRIFICATIONS STRUCTURELLES FINALES
    # -----------------------------------------------------------------------------
    print(f"--- ÉTAPE 4 : CONTRÔLES ET SÉCURITÉ GÉOSPATIALE ---")
    with rasterio.open(output_raster) as img_verif:
        print(f"  [VÉRIFICATION] Nombre de bandes générées : {img_verif.count}")
        
        print("\n--- ÉTAPE 5 : STATISTIQUES DESCRIPTIVES DU RASTER AGRÉGÉ ---")
        print("  Répartition de la variable 'classe_dominante' (Bande 1) :")
        for classe_id, total_pixels in dict_repartition_classes.items():
            if total_pixels > 0:
                print(f"    - Classe {classe_id} : {total_pixels} pixels")
            
        moyenne_spatiale_c2 = (somme_c2 / nb_valides_c2) if nb_valides_c2 > 0 else 0.0
        moyenne_spatiale_c3 = (somme_c3 / nb_valides_c3) if nb_valides_c3 > 0 else 0.0
        
        print(f"  Moyenne spatiale de 'prop_Q4' (Bande 2) : {moyenne_spatiale_c2:.4f}")
        print(f"  Moyenne spatiale de 'prop_Q5' (Bande 3) : {moyenne_spatiale_c3:.4f}")
        
    print(f"[Succès] Fichier sauvegardé sans aucune perte de bloc : {output_raster.resolve()}\n")

# ==========================================
# EXÉCUTION DYNAMIQUE
# ==========================================
if __name__ == "__main__":
    # Paramétrage des résolutions initiale et finale (ex: 10m à 1000m / 1km)
    resinit = 10
    resfin = 1000
    
    dir_input = Path("../geodata_outputs")
    dir_output = Path("../geodata_outputs")
    
    dir_output.mkdir(parents=True, exist_ok=True)
    
    # Configuration des répertoires pour le fichier de déforestation
    input_file = dir_input / "deforest_2020_2021.tif"
    output_file = dir_output / "deforest_2020_2021_redim_1km_class_propDefor.tif"
    
    if input_file.exists():
        aggregate_raster_by_quintiles(input_file, output_file, resinit, resfin)
    else:
        print(f"[Erreur] Le fichier d'entrée spécifié est introuvable : {input_file.resolve()}")


[Début] Traitement optimisé du fichier : deforest_2020_2021.tif
  Facteur d'agrégation calculé : 100 (Matrice locale de 100x100 pixels)
--- ÉTAPE 1 : COLLECTE DES STATISTIQUES GLOBALES (PASSE 1/2) ---
  Dimensions d'origine : 7928x8393
  Dimensions utiles retenues (multiples de 100) : 7900x8300
  Bordure éliminée au départ (sans perte de blocs internes) : 28px en X, 93px en Y
  Statistiques globales estimées sur la zone utile :
    - Minimum : -0.9260
    - Maximum : 1.1046
    - Moyenne : 0.0036
--- ÉTAPE 2 : SEUILS DES QUINTILES GLOBAUX ---
  [OK] Confirmation : Aucun NaN détecté dans les seuils.
  Quintile 1 (0-20%)  : <= -0.0181
  Quintile 2 (20-40%) : > -0.0181 et <= -0.0044
  Quintile 3 (40-60%) : > -0.0044 et <= 0.0072
  Quintile 4 (60-80%) : > 0.0072 et <= 0.0226
  Quintile 5 (80-100%): > 0.0226
--- ÉTAPE 3 : AGRÉGATION MULTI-RÉSOLUTION BLOC PAR BLOC ---
--- ÉTAPE 4 : CONTRÔLES ET SÉCURITÉ GÉOSPATIALE ---
  [VÉRIFICATION] Nombre de bandes générées : 3

--- ÉTAPE 5 : STATISTIQU